In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pylab as pl
import scipy.fft as fft
from scipy.ndimage import sobel
from scipy.interpolate import RectBivariateSpline
import matplotlib.animation as animation
from scipy.interpolate import UnivariateSpline
from scipy.optimize import minimize_scalar
from scipy.optimize import minimize

# Load Needed Data
orbits = np.loadtxt('./out/tracer.out')
R      = np.loadtxt('R.dat')
Z      = np.loadtxt('Z.dat')
psi    = np.loadtxt('psi_p.dat')

for i in range (1,9999999):
    if orbits[i,1] <= orbits[i-1,1]:
        break
number_of_particles=i

# Set Divertor
div_size=np.linspace(0,200,201)
r_1,r_2=1.23,1.48
z_1,z_2=-1.4,-1.2
m=(z_2-z_1)/(r_2-r_1)
b=z_1-m*r_1
x_val=[r_1,r_2]
y_val=[m * x + b for x in x_val]

# Set Variables
divpsi=[]

# Calculate the gradients of the poloidal flux function
eq_psi_rz=-psi+np.max(psi)
dpsi_dR = sobel(eq_psi_rz, axis=1) / (R[-1] - R[0])
dpsi_dZ = sobel(eq_psi_rz, axis=0) / (Z[-1] - Z[0])

# Interpolate the gradients
dpsi_dR_interp = RectBivariateSpline(Z, R, dpsi_dR)
dpsi_dZ_interp = RectBivariateSpline(Z, R, dpsi_dZ)

# Define a function to compute the gradient magnitude at a given point (R, Z)
def gradient_magnitude(point):
    dR, dZ = dpsi_dR_interp(point[1], point[0]), dpsi_dZ_interp(point[1], point[0])
    return np.sqrt(dR**2 + dZ**2)[0, 0]  # Take the first element of the array
    
# Define an objective function to minimize the gradient magnitude
def objective_function(point):
    return gradient_magnitude(point)
    
# Find the X-point by minimizing the gradient magnitude
# You can initialize the search for the minimum with a rough estimate of the X-point location
initial_guess = (1.3, -1.2)  # Adjust this value based on your specific case
bounds = [(1.2, 1.4), (-1.4, -1)]  # Set the bounds for the search
result = minimize(objective_function, initial_guess, bounds=bounds, method='L-BFGS-B')

R_x, Z_x      = result.x
R_idxx = np.abs(R - R_x).argmin()
Z_idxx = np.abs(Z - Z_x).argmin()
    
# Obtain psi value at the X-point
psi_at_xpoint = eq_psi_rz[Z_idxx, R_idxx]
print(f"X-point coordinates: R = {R_x}, Z = {Z_x}")
print(f"psi at the X-point: psi = {psi_at_xpoint}")

#Extract R,Z Parameterization for lcfs
contour = plt.contour(R,Z,eq_psi_rz, levels=[psi_at_xpoint])
    
# Find the path that includes the X-point coordinates
contour_paths = contour.collections[0].get_paths()
cont          = contour_paths[0].vertices

R_contour, Z_contour = cont[:,0], cont[:,1]

sepdiv = []
for vert in range(len(R_contour)):
    sepdiv.append(abs(m*R_contour[vert] - Z_contour[vert] + b))
sepinx = np.argmin(sepdiv)
dpsidr = np.gradient(R_contour)[sepinx]
dpsidz = np.gradient(Z_contour)[sepinx]
gradpsi  = np.sqrt(dpsidr**2+dpsidz**2)

start_time_step=1
end_time_step=5000

time = np.linspace(start_time_step,end_time_step,end_time_step-1)

plt.figure(figsize=(5, 9))
plt.contour(R,Z,psi,50,alpha=0.15)

for particle_ID in range (0,number_of_particles-1):   ###herre choose the particles you want to plot
    start=(start_time_step-1)*number_of_particles+particle_ID
    end=(end_time_step-1)*number_of_particles+particle_ID

    plt.plot(orbits[start:end:number_of_particles,2],orbits[start:end:number_of_particles,3])
    
    xloc = orbits[start:end:number_of_particles,2]
    yloc = orbits[start:end:number_of_particles,3]
    poi  = abs(m*xloc + b - yloc)
    inrz = np.argmin(poi)
    Rv   = np.argmin(R-xloc[inrz])
    Zv   = np.argmin(Z-yloc[inrz])
    divpsi.append(eq_psi_rz[Rv][Zv])
        
#Calculate SOL width

delpsi  = np.max(divpsi) - np.min(divpsi)
gradpsi = abs(gradpsi)
lamq    = delpsi/gradpsi
    
#Calder's Divertor Plot Start
plt.plot(x_val,y_val,color='black',linewidth=3.0)
#Calder's Divertor Plot End

plt.xticks(fontsize=15)
plt.yticks(fontsize=15)
plt.title('Tracer particle orbits', fontsize=15)
plt.xlabel('R(m)', fontsize=20)
plt.ylabel('Z(m)',fontsize=20)
plt.axis('scaled')
plt.xlim(min(R), max(R))
plt.ylim(min(Z), max(Z))
# Save the plot as a PNG file
plt.savefig('./out/tracer_particle_orbits_dt10_nm4000.png')
plt.show()

print('SOL Width:')
print(lamq)